# Swap规则置换分析报告样例

本示例演示如何使用 `swap_analysis_report` 函数进行规则置换风险评估。

## 业务场景

在金融信贷风控中，**策略通常以拒绝规则的形式定义**：
- 规则命中（1）= 拒绝
- 规则不命中（0）= 通过

### 规则定义

**原策略拒绝规则**：`青云24 < 650`
- 青云24分数低于650分的客户被原策略拒绝

**新策略拒绝规则**：`百融定制分V9 < 680 | 青云24 < 550`
- 百融定制分V9低于680分 或 青云24低于550分的客户被新策略拒绝

### Swap四象限分析

| 象限 | 原策略 | 新策略 | 说明 |
|------|--------|--------|------|
| in-in | 通过 | 通过 | 保留样本（青云24 >= 650 且不满足新拒绝规则）|
| in-out | 通过 | 拒绝 | 置出样本（650 > 青云24 >= 550 且百融分 < 680）|
| out-in | 拒绝 | 通过 | 置入样本（核心关注，550 <= 青云24 < 650）|
| out-out | 拒绝 | 拒绝 | 仍拒绝样本（青云24 < 550）|

### 评分说明

**用于坏账预估的评分**：`中智小牛分C3`
- 值域非[0,1]：分数越大，风险越低
- 高分客户风险低，低分客户风险高

In [1]:
import sys
sys.path.insert(0, '/Users/xiaoxi/CodeBuddy/hscredit/hscredit')

import pandas as pd
import numpy as np
from hscredit import Rule
from hscredit.report.swap_analysis_report import (
    create_swap_dataset_from_rules,
    swap_analysis_report,
    SwapType
)

## 1. 数据准备

In [2]:
# 加载示例数据
df = pd.read_excel('../utils/hscredit.xlsx')

# 创建评分：中智小牛分C3（分数越大风险越低）
df['score'] = df['中智小牛分C3']

# 创建模拟金额字段
np.random.seed(42)
df['amount'] = np.random.uniform(5000, 50000, len(df)).round(2)

print(f"数据形状: {df.shape}")
print(f"\n评分统计（中智小牛分C3）:")
print(df['score'].describe())

数据形状: (22729, 12)

评分统计（中智小牛分C3）:
count    22715.000000
mean       615.121594
std         96.148123
min        300.000000
25%        550.000000
50%        612.000000
75%        677.000000
max        850.000000
Name: score, dtype: float64


## 2. 定义拒绝规则

**原策略拒绝规则**：`青云24 < 650`
- 青云24低于650分的客户被拒绝

**新策略拒绝规则**：`百融定制分V9 < 680 | 青云24 < 550`
- 百融定制分V9低于680分 或 青云24低于550分的客户被拒绝

In [3]:
# 定义原策略拒绝规则：青云24 < 650
original_reject_rule = Rule('青云24 < 650')

# 定义新策略拒绝规则：百融定制分V9 < 680 | 青云24 < 550
new_reject_rule = Rule('百融定制分V9 < 680 | 青云24 < 550')

print("✅ 规则定义完成")

✅ 规则定义完成


## 3. 创建Swap数据集

In [4]:
# 从Rule对象创建swap数据集
swap_df = create_swap_dataset_from_rules(
    df,
    original_rule=original_reject_rule,    # 原策略Rule对象
    new_rule=new_reject_rule,              # 新策略Rule对象
    score_col='score',
    amount_col='amount',
    rule_type='reject'                     # 拒绝规则模式
)

print("Swap四象限分布（拒绝规则模式）：")
swap_counts = swap_df['swap_type'].value_counts()
for swap_type, count in swap_counts.items():
    pct = count / len(swap_df) * 100
    print(f"  {swap_type}: {count} ({pct:.2f}%)")

print(f"\n总样本数: {len(swap_df)}")

Swap四象限分布（拒绝规则模式）：
  out-in: 9619 (42.32%)
  out-out: 7782 (34.24%)
  in-in: 4776 (21.01%)
  in-out: 552 (2.43%)

总样本数: 22729


## 4. 单标签Swap分析（使用target参数）

使用 `swap_analysis_report` 函数进行完整的swap分析。

In [5]:
# 创建目标变量
df['target_dpd15'] = (df['MOB1'] > 15).astype(int)

# 准备参考数据
ref_df = df[['score', 'target_dpd15', 'amount']].sample(frac=0.7, random_state=42)

# 单标签swap分析
result_single = swap_analysis_report(
    swap_df=swap_df,
    reference_df=ref_df,
    score_col='score',
    target='target_dpd15',    # 单标签模式
    amount_col='amount',      # 金额口径
    out_in_uplift=2.0         # out-in样本风险上浮2倍
)

print("✅ 单标签分析完成")

✅ 单标签分析完成


In [6]:
# 查看汇总报告
print("=" * 120)
print("单标签分析 - 订单口径汇总报告")
print("=" * 120)
result_single.summary_report_count

单标签分析 - 订单口径汇总报告


,阶段,样本数,样本占比,坏样本数,坏样本率,LIFT,风险改善,风险拒绝比,原通过率,新通过率,通过率变化(绝对),通过率变化(相对),坏账率变化(绝对),坏账率变化(相对)
0,原策略通过,5328.0,23.44%,324.0,0.0608,0.5970,0.4030,1.7194,23.44%,NaN,-,-,-0.0411,-40.30%
1,新策略置出(in-out),552.0,2.43%,39.0,0.0706,0.6935,0.3065,12.6198,-,NaN,-,-,-0.0312,-30.65%
2,原策略保留(in-in),4776.0,21.01%,285.0,0.0597,0.5858,0.4142,1.9712,-,NaN,-,-,-0.0422,-41.42%
3,新策略置入(out-in),9619.0,42.32%,1367.9,0.1422,1.3961,-0.3961,-0.9359,-,NaN,-,-,0.0403,39.61%
4,新策略通过,14395.0,63.33%,1652.9,0.1148,1.1272,-0.1272,-0.2009,NaN,63.33%,39.89%,170.18%,0.0130,12.72%
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,原策略拒绝,17401.0,76.56%,1991.3,0.1144,1.1234,-0.1234,-0.1612,23.44%,NaN,-,-,0.0126,12.34%
7,新策略置入(out-in),9619.0,42.32%,1367.9,0.1422,1.3961,-0.3961,-0.9359,-,NaN,-,-,0.0403,39.61%
8,原策略保留(in-in),4776.0,21.01%,285.0,0.0597,0.5858,0.4142,1.9712,-,NaN,-,-,-0.0422,-41.42%
9,新策略置出(in-out),552.0,2.43%,39.0,0.0706,0.6935,0.3065,12.6198,-,NaN,-,-,-0.0312,-30.65%


## 5. 多标签Swap分析（使用overdue+dpds参数）

支持同时分析多个逾期标签，使用方式与 `feature_bin_stats` 一致。

In [19]:
# 多标签swap分析
result = swap_analysis_report(
    swap_df=swap_df,
    reference_df=df[['score', 'MOB1', 'amount']],  # 原始数据，函数内部会生成标签
    score_col='score',
    overdue='MOB1',           # 逾期天数字段
    dpds=[15, 30],            # 逾期定义天数列表
    amount_col='amount',      # 金额口径
    out_in_uplift=2.0,        # out-in风险上浮因子
    target_aliases={          # 目标变量别名，用于报告展示
        'MOB1_15+': 'DPD15+',
        'MOB1_30+': 'DPD30+'
    }
)

print("✅ 多标签分析完成")
print(f"分析的标签: {result.targets}")

✅ 多标签分析完成
分析的标签: ['MOB1_15+', 'MOB1_30+']


## 6. DPD15+ 标签分析结果

In [21]:
# 获取DPD15+的订单口径报告
print("=" * 120)
print("DPD15+ 订单口径汇总报告（含通过率和风险变化）")
print("=" * 120)
report_dpd15_count = result.get_summary_report('count', target='MOB1_15+')
report_dpd15_count

DPD15+ 订单口径汇总报告（含通过率和风险变化）


,阶段,样本数,样本占比,坏样本数,坏样本率,LIFT,风险改善,风险拒绝比,原通过率,新通过率,通过率变化(绝对),通过率变化(相对),坏账率变化(绝对),坏账率变化(相对)
0,原策略通过,5328.0,23.44%,314.2,0.0590,0.5901,0.4099,1.7486,23.44%,NaN,-,-,-0.0410,-40.99%
1,新策略置出(in-out),552.0,2.43%,38.7,0.0701,0.7014,0.2986,12.2939,-,NaN,-,-,-0.0298,-29.86%
2,原策略保留(in-in),4776.0,21.01%,275.5,0.0577,0.5772,0.4228,2.0119,-,NaN,-,-,-0.0422,-42.28%
3,新策略置入(out-in),9619.0,42.32%,1338.5,0.1392,1.3925,-0.3925,-0.9275,-,NaN,-,-,0.0392,39.25%
4,新策略通过,14395.0,63.33%,1614.0,0.1121,1.1220,-0.1220,-0.1927,NaN,63.33%,39.89%,170.18%,0.0122,12.20%
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,原策略拒绝,17401.0,76.56%,1957.0,0.1125,1.1255,-0.1255,-0.1639,23.44%,NaN,-,-,0.0125,12.55%
7,新策略置入(out-in),9619.0,42.32%,1338.5,0.1392,1.3925,-0.3925,-0.9275,-,NaN,-,-,0.0392,39.25%
8,原策略保留(in-in),4776.0,21.01%,275.5,0.0577,0.5772,0.4228,2.0119,-,NaN,-,-,-0.0422,-42.28%
9,新策略置出(in-out),552.0,2.43%,38.7,0.0701,0.7014,0.2986,12.2939,-,NaN,-,-,-0.0298,-29.86%


In [9]:
# 获取DPD15+的金额口径报告
print("=" * 120)
print("DPD15+ 金额口径汇总报告")
print("=" * 120)
report_dpd15_amount = result.get_summary_report('amount', target='MOB1_15+')
report_dpd15_amount

DPD15+ 金额口径汇总报告


,阶段,样本数,样本占比,坏样本数,坏样本率,LIFT,风险改善,风险拒绝比,原通过率,新通过率,通过率变化(绝对),通过率变化(相对),坏账率变化(绝对),坏账率变化(相对)
0,原策略通过,1.459714e+08,23.31%,8627057.2,0.0591,0.5912,0.4088,1.7536,23.44%,NaN,-,-,-0.0409,-40.88%
1,新策略置出(in-out),1.481641e+07,2.37%,1047742.0,0.0707,0.7074,0.2926,12.3667,-,NaN,-,-,-0.0293,-29.26%
2,原策略保留(in-in),1.311550e+08,20.95%,7579315.2,0.0578,0.5781,0.4219,2.0144,-,NaN,-,-,-0.0422,-42.19%
3,新策略置入(out-in),2.653240e+08,42.37%,36909544.8,0.1391,1.3916,-0.3916,-0.9241,-,NaN,-,-,0.0391,39.16%
4,新策略通过,3.964791e+08,63.32%,44488860.0,0.1122,1.1225,-0.1225,-0.1934,NaN,63.33%,39.89%,170.18%,0.0122,12.25%
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,原策略拒绝,4.801858e+08,76.69%,53968842.3,0.1124,1.1243,-0.1243,-0.1620,23.44%,NaN,-,-,0.0124,12.43%
7,新策略置入(out-in),2.653240e+08,42.37%,36909544.8,0.1391,1.3916,-0.3916,-0.9241,-,NaN,-,-,0.0391,39.16%
8,原策略保留(in-in),1.311550e+08,20.95%,7579315.2,0.0578,0.5781,0.4219,2.0144,-,NaN,-,-,-0.0422,-42.19%
9,新策略置出(in-out),1.481641e+07,2.37%,1047742.0,0.0707,0.7074,0.2926,12.3667,-,NaN,-,-,-0.0293,-29.26%


### 6.1 DPD15+ 通过率分析

In [22]:
print("=" * 80)
print("DPD15+ 通过率分析")
print("=" * 80)

pra = result.pass_rate_analysis
print(f"\n原策略通过率: {pra.original_pass_rate:.2%}")
print(f"新策略通过率: {pra.new_pass_rate:.2%}")
print(f"通过率变化(绝对): {pra.absolute_change:.2%}")
print(f"通过率变化(相对): {pra.relative_change_pct:.2f}%")

result.pass_rate_report

DPD15+ 通过率分析

原策略通过率: 23.44%
新策略通过率: 63.33%
通过率变化(绝对): 39.89%
通过率变化(相对): 170.18%


,原策略通过率,新策略通过率,通过率变化(绝对),通过率变化(相对)
0,23.44%,63.33%,39.89%,170.18%


### 6.2 DPD15+ 风险拒绝率分析

In [23]:
print("=" * 80)
print("DPD15+ 风险拒绝率分析")
print("=" * 80)

rrm = result.risk_rejection_metrics
print(f"\n原坏账率: {rrm.bad_rate_before:.4f}")
print(f"新坏账率: {rrm.bad_rate_after:.4f}")
print(f"坏账率改善(绝对): {rrm.bad_rate_improvement:.4f}")
print(f"坏账率改善(相对): {rrm.bad_rate_relative_improvement:.2f}%")
print(f"\n拒绝率: {rrm.rejection_rate:.2%}")
print(f"风险拒绝率: {rrm.risk_rejection_rate:.4f}")
print(f"  (每拒绝1%样本，坏账率降低{rrm.risk_rejection_rate:.4f})")
print(f"\n通过率下降1%的坏账改善: {rrm.pass_rate_drop_1pct_improvement:.4f}")

result.risk_rejection_report

DPD15+ 风险拒绝率分析

原坏账率: 0.0590
新坏账率: 0.1121
坏账率改善(绝对): -0.0532
坏账率改善(相对): -90.14%

拒绝率: 36.67%
风险拒绝率: -0.1450
  (每拒绝1%样本，坏账率降低-0.1450)

通过率下降1%的坏账改善: 0.0000


,拒绝率,风险拒绝率,通过率下降1%的坏账改善,原坏账率,新坏账率,坏账率改善(绝对),坏账率改善(相对)
0,36.67%,-0.1450,0.0000,0.0590,0.1121,-0.0532,-90.14%


## 7. DPD30+ 标签分析结果

In [24]:
# 获取DPD30+的订单口径报告
print("=" * 120)
print("DPD30+ 订单口径汇总报告")
print("=" * 120)
report_dpd30_count = result.get_summary_report('count', target='MOB1_30+')
report_dpd30_count

DPD30+ 订单口径汇总报告


,阶段,样本数,样本占比,坏样本数,坏样本率,LIFT,风险改善,风险拒绝比,原通过率,新通过率,通过率变化(绝对),通过率变化(相对),坏账率变化(绝对),坏账率变化(相对)
0,原策略通过,5328.0,23.44%,211.4,0.0397,0.5758,0.4242,1.8096,23.44%,NaN,-,-,-0.0292,-42.42%
1,新策略置出(in-out),552.0,2.43%,26.6,0.0482,0.6998,0.3002,12.3620,-,NaN,-,-,-0.0207,-30.02%
2,原策略保留(in-in),4776.0,21.01%,184.8,0.0387,0.5615,0.4385,2.0870,-,NaN,-,-,-0.0302,-43.85%
3,新策略置入(out-in),9619.0,42.32%,920.8,0.0957,1.3893,-0.3893,-0.9198,-,NaN,-,-,0.0268,38.93%
4,新策略通过,14395.0,63.33%,1105.6,0.0768,1.1146,-0.1146,-0.1810,NaN,63.33%,39.89%,170.18%,0.0079,11.46%
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,原策略拒绝,17401.0,76.56%,1354.8,0.0779,1.1299,-0.1299,-0.1697,23.44%,NaN,-,-,0.0089,12.99%
7,新策略置入(out-in),9619.0,42.32%,920.8,0.0957,1.3893,-0.3893,-0.9198,-,NaN,-,-,0.0268,38.93%
8,原策略保留(in-in),4776.0,21.01%,184.8,0.0387,0.5615,0.4385,2.0870,-,NaN,-,-,-0.0302,-43.85%
9,新策略置出(in-out),552.0,2.43%,26.6,0.0482,0.6998,0.3002,12.3620,-,NaN,-,-,-0.0207,-30.02%


In [25]:
# 获取DPD30+的金额口径报告
print("=" * 120)
print("DPD30+ 金额口径汇总报告")
print("=" * 120)
report_dpd30_amount = result.get_summary_report('amount', target='MOB1_30+')
report_dpd30_amount

DPD30+ 金额口径汇总报告


,阶段,样本数,样本占比,坏样本数,坏样本率,LIFT,风险改善,风险拒绝比,原通过率,新通过率,通过率变化(绝对),通过率变化(相对),坏账率变化(绝对),坏账率变化(相对)
0,原策略通过,1.459714e+08,23.31%,5807332.4,0.0398,0.5770,0.4230,1.8145,23.44%,NaN,-,-,-0.0292,-42.30%
1,新策略置出(in-out),1.481641e+07,2.37%,721184.9,0.0487,0.7060,0.2940,12.4266,-,NaN,-,-,-0.0203,-29.40%
2,原策略保留(in-in),1.311550e+08,20.95%,5086147.5,0.0388,0.5624,0.4376,2.0890,-,NaN,-,-,-0.0302,-43.76%
3,新策略置入(out-in),2.653240e+08,42.37%,25397595.1,0.0957,1.3883,-0.3883,-0.9164,-,NaN,-,-,0.0268,38.83%
4,新策略通过,3.964791e+08,63.32%,30483742.6,0.0769,1.1151,-0.1151,-0.1818,NaN,63.33%,39.89%,170.18%,0.0079,11.51%
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,原策略拒绝,4.801858e+08,76.69%,37365308.0,0.0778,1.1286,-0.1286,-0.1677,23.44%,NaN,-,-,0.0089,12.86%
7,新策略置入(out-in),2.653240e+08,42.37%,25397595.1,0.0957,1.3883,-0.3883,-0.9164,-,NaN,-,-,0.0268,38.83%
8,原策略保留(in-in),1.311550e+08,20.95%,5086147.5,0.0388,0.5624,0.4376,2.0890,-,NaN,-,-,-0.0302,-43.76%
9,新策略置出(in-out),1.481641e+07,2.37%,721184.9,0.0487,0.7060,0.2940,12.4266,-,NaN,-,-,-0.0203,-29.40%


## 8. 订单口径 vs 金额口径对比

In [26]:
print("=" * 120)
print("DPD15+ 订单口径 vs 金额口径对比")
print("=" * 120)

# 提取关键列对比
compare_cols = ['阶段', '样本数', '样本占比', '坏样本率', 'LIFT', '风险改善', '风险拒绝比',
                '通过率变化(绝对)', '通过率变化(相对)', '坏账率变化(绝对)', '坏账率变化(相对)']

print("\n订单口径（样本数）：")
print(report_dpd15_count[compare_cols].to_string())

print("\n金额口径（金额和）：")
print(report_dpd15_amount[compare_cols].to_string())

DPD15+ 订单口径 vs 金额口径对比

订单口径（样本数）：
               阶段      样本数     样本占比    坏样本率    LIFT     风险改善    风险拒绝比 通过率变化(绝对) 通过率变化(相对) 坏账率变化(绝对) 坏账率变化(相对)
0           原策略通过   5328.0   23.44%  0.0590  0.5901   0.4099   1.7486         -         -   -0.0410   -40.99%
1   新策略置出(in-out)    552.0    2.43%  0.0701  0.7014   0.2986  12.2939         -         -   -0.0298   -29.86%
2    原策略保留(in-in)   4776.0   21.01%  0.0577  0.5772   0.4228   2.0119         -         -   -0.0422   -42.28%
3   新策略置入(out-in)   9619.0   42.32%  0.1392  1.3925  -0.3925  -0.9275         -         -    0.0392    39.25%
4           新策略通过  14395.0   63.33%  0.1121  1.1220  -0.1220  -0.1927    39.89%   170.18%    0.0122    12.20%
5             NaN      NaN      NaN     NaN     NaN      NaN      NaN       NaN       NaN       NaN       NaN
6           原策略拒绝  17401.0   76.56%  0.1125  1.1255  -0.1255  -0.1639         -         -    0.0125    12.55%
7   新策略置入(out-in)   9619.0   42.32%  0.1392  1.3925  -0.3925  -0.9275         -       

## 9. 多标签关键指标对比

In [27]:
# 获取DPD30+的分析指标
rrm_dpd30 = result.risk_rejection_metrics_dict['MOB1_30+']

# 构建对比表
comparison = pd.DataFrame({
    '指标': [
        '新策略通过率', '通过率变化(绝对)', '通过率变化(相对)',
        '原坏账率', '新坏账率', '坏账率改善(绝对)', '坏账率改善(相对)',
        '拒绝率', '风险拒绝率', '通过率↓1%的坏账改善'
    ],
    'DPD15+': [
        f"{result.pass_rate_analysis.new_pass_rate:.2%}",
        f"{result.pass_rate_analysis.absolute_change:.2%}",
        f"{result.pass_rate_analysis.relative_change_pct:.2f}%",
        f"{result.risk_rejection_metrics.bad_rate_before:.4f}",
        f"{result.risk_rejection_metrics.bad_rate_after:.4f}",
        f"{result.risk_rejection_metrics.bad_rate_improvement:.4f}",
        f"{result.risk_rejection_metrics.bad_rate_relative_improvement:.2f}%",
        f"{result.risk_rejection_metrics.rejection_rate:.2%}",
        f"{result.risk_rejection_metrics.risk_rejection_rate:.4f}",
        f"{result.risk_rejection_metrics.pass_rate_drop_1pct_improvement:.4f}",
    ],
    'DPD30+': [
        f"{result.pass_rate_analysis.new_pass_rate:.2%}",
        f"{result.pass_rate_analysis.absolute_change:.2%}",
        f"{result.pass_rate_analysis.relative_change_pct:.2f}%",
        f"{rrm_dpd30.bad_rate_before:.4f}",
        f"{rrm_dpd30.bad_rate_after:.4f}",
        f"{rrm_dpd30.bad_rate_improvement:.4f}",
        f"{rrm_dpd30.bad_rate_relative_improvement:.2f}%",
        f"{rrm_dpd30.rejection_rate:.2%}",
        f"{rrm_dpd30.risk_rejection_rate:.4f}",
        f"{rrm_dpd30.pass_rate_drop_1pct_improvement:.4f}",
    ],
})

print("=" * 80)
print("多标签关键指标对比")
print("=" * 80)
comparison

多标签关键指标对比


,指标,DPD15+,DPD30+
0,新策略通过率,63.33%,63.33%
1,通过率变化(绝对),39.89%,39.89%
2,通过率变化(相对),170.18%,170.18%
3,原坏账率,0.0590,0.0397
4,新坏账率,0.1121,0.0768
5,坏账率改善(绝对),-0.0532,-0.0371
6,坏账率改善(相对),-90.14%,-93.58%
7,拒绝率,36.67%,36.67%
8,风险拒绝率,-0.1450,-0.1013
9,通过率↓1%的坏账改善,0.0000,0.0000


## 10. 详细四象限报告（DPD15+）

In [28]:
print("=" * 100)
print("详细四象限报告 - DPD15+ 订单口径")
print("=" * 100)

detail_report = result.get_detail_report('count', target='MOB1_15+')
detail_report

详细四象限报告 - DPD15+ 订单口径


,Swap类型,说明,样本数,样本占比,坏样本率(DPD15+),风险上浮
0,in-in,原策略通过，新策略通过,4776,21.01%,0.0577,-
1,in-out,原策略通过，新策略拒绝,552,2.43%,0.0701,-
2,out-in,原策略拒绝，新策略通过（置入）,9619,42.32%,0.1392,2.0x
3,out-out,原策略拒绝，新策略拒绝,7782,34.24%,0.0795,-


In [29]:
print("=" * 100)
print("详细四象限报告 - DPD15+ 金额口径")
print("=" * 100)

detail_report_amount = result.get_detail_report('amount', target='MOB1_15+')
detail_report_amount

详细四象限报告 - DPD15+ 金额口径


,Swap类型,说明,样本数,样本占比,坏样本率(DPD15+),风险上浮
0,in-in,原策略通过，新策略通过,1.311550e+08,20.95%,0.0578,-
1,in-out,原策略通过，新策略拒绝,1.481641e+07,2.37%,0.0707,-
2,out-in,原策略拒绝，新策略通过（置入）,2.653240e+08,42.37%,0.1391,2.0x
3,out-out,原策略拒绝，新策略拒绝,2.148617e+08,34.31%,0.0794,-


## 11. 人工传入原策略通过率场景

当数据中没有out-out样本（原策略拒绝且新策略也拒绝）时，无法自动计算原策略通过率，
需要人工传入原策略通过率进行分析。

In [30]:
print("=" * 80)
print("场景：无out-out样本，人工传入通过率")
print("=" * 80)

# 模拟只有in-in, in-out, out-in的情况（假设没有out-out）
swap_partial = swap_df[swap_df['swap_type'].isin(['in-in', 'in-out', 'out-in'])].copy()

print(f"部分样本数: {len(swap_partial)}")
print("\nSwap类型分布:")
for swap_type, count in swap_partial['swap_type'].value_counts().items():
    print(f"  {swap_type}: {count}")

# 假设原策略通过率为历史经验值
assumed_original_pass_rate = 0.65

# 重新分析（人工传入原策略通过率）
result_partial = swap_analysis_report(
    swap_df=swap_partial,
    reference_df=df[['score', 'MOB1', 'amount']],
    score_col='score',
    overdue='MOB1',
    dpds=15,
    amount_col='amount',
    out_in_uplift=2.0,
    original_pass_rate=assumed_original_pass_rate,
    target_aliases={'MOB1_15+': 'DPD15+'}
)

print(f"\n传入原策略通过率: {assumed_original_pass_rate:.2%}")
print(f"计算得到的新通过率: {result_partial.pass_rate_analysis.new_pass_rate:.2%}")
print(f"通过率变化(绝对): {result_partial.pass_rate_analysis.absolute_change:.2%}")
print(f"通过率变化(相对): {result_partial.pass_rate_analysis.relative_change_pct:.2f}%")

print("\n汇总报告:")
result_partial.get_summary_report('count', target='MOB1_15+')

场景：无out-out样本，人工传入通过率
部分样本数: 14947

Swap类型分布:
  out-in: 9619
  in-in: 4776
  in-out: 552

传入原策略通过率: 65.00%
计算得到的新通过率: 96.31%
通过率变化(绝对): 31.31%
通过率变化(相对): 48.16%

汇总报告:


,阶段,样本数,样本占比,坏样本数,坏样本率,LIFT,风险改善,风险拒绝比,原通过率,新通过率,通过率变化(绝对),通过率变化(相对),坏账率变化(绝对),坏账率变化(相对)
0,原策略通过,5328,35.65%,230.9,0.0433,0.5946,0.4054,1.1373,65.00%,NaN,-,-,-0.0295,-40.54%
1,新策略置出(in-out),552,3.69%,24.9,0.0451,0.6186,0.3814,10.3276,-,NaN,-,-,-0.0278,-38.14%
2,原策略保留(in-in),4776,31.95%,206.0,0.0431,0.5918,0.4082,1.2775,-,NaN,-,-,-0.0298,-40.82%
3,新策略置入(out-in),9619,64.35%,843.2,0.0877,1.2027,-0.2027,-0.3149,-,NaN,-,-,0.0148,20.27%
4,新策略通过,14395,96.31%,1049.2,0.0729,1.0000,0.0000,0.0000,NaN,96.31%,31.31%,48.16%,0.0000,0.00%


## 12. 业务解读

### 拒绝规则 vs 通过规则

在风控场景中，策略通常定义为**拒绝规则**（而非通过规则）：
- **拒绝规则**：规则命中 = 拒绝，规则不命中 = 通过
- **通过规则**：规则命中 = 通过，规则不命中 = 拒绝

### 本例中的规则逻辑

**原策略拒绝规则**：`青云24 < 650`
- 青云分低于650分的客户被拒绝，高于等于650分的客户通过

**新策略拒绝规则**：`百融定制分V9 < 680 | 青云24 < 550`
- 百融定制分V9低于680分 或 青云24低于550分的客户被拒绝
- 这意味着：
  - [550, 650)区间的客户：原策略拒绝 → 新策略通过（**置入**）
  - 百融分<680且青云分>=650的客户：原策略通过 → 新策略拒绝（**置出**）

### 报告指标说明

#### 通过率变化
- **原通过率**：原策略通过率
- **新通过率**：新策略通过率
- **通过率变化(绝对)**：新通过率 - 原通过率（百分点）
- **通过率变化(相对)**：(新-旧)/|旧| × 100%（百分比）

#### 风险变化
- **坏账率变化(绝对)**：当前阶段坏账率 - 整体坏账率
- **坏账率变化(相对)**：(当前-整体)/整体 × 100%（百分比）
- **坏账率改善(绝对)**：原坏账率 - 新坏账率
- **坏账率改善(相对)**：改善幅度/原坏账率 × 100%（百分比）

#### 其他指标
- **LIFT**：当前分组坏账率 / 整体坏账率
- **风险改善**：(整体坏账率 - 当前坏账率) / 整体坏账率
- **风险拒绝比**：风险改善 / 样本占比
- **风险拒绝率**：坏账率改善 / 拒绝率

### 决策建议

- 如果**风险拒绝比** > 1：说明拒绝这部分样本的收益大于成本，建议收紧
- 如果**风险拒绝比** < 1：说明拒绝这部分样本的收益小于成本，建议放宽
- 重点关注**out-in**（置入样本）的风险，这部分是新增的风险敞口